# 2-D：雙核 demo，外面鎖住順序、裡面放手判斷

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/notebooks/2-D_dual_core.ipynb)

> 📍 **對應簡報**：第 2 堂「第三站：資安與雙核」，投影片 **24–27**（課堂時間 1:06–1:15，講師示範）

## 這本 notebook 在做什麼

把上週和今天學的東西接起來，就是課名裡的「**雙拼**」：

- **外面**是 1-A 的 workflow（順序鎖死）：一定先檢查語法、一定檢查格式，不合格退回重寫。
- **裡面**的 `give_feedback` 是 1-B 的小 agent（放手判斷）：
  手冊用 2-A 的「溫和鼓勵」或「嚴格審查」，工具箱裡有 2-B 的命名檢查腳本、`run_code`、`read_spec`，
  要不要跑命名檢查、要不要讀題目，由它自己決定。
- 兩個新零件一句話帶過：`skill_loader` 節點看所有手冊的**名片**挑一本；腳本外面包一層 `@tool` 就進了工具箱。
- 最後用 **checkpointer ＋ thread_id** 把每一步的 State 存起來：中斷了也能接著跑。

| 段落 | 你會看到什麼 | 對應投影片 |
| --- | --- | --- |
| 第 1 段 | 寫出 `dual_core.py`，畫出雙核的流程圖 | 24 |
| 第 2 段 | 完整跑一次：外層一站一站走，裡面的 agent 自己選工具 | 24–25 |
| 第 3 段 | 中斷：跑到一半先停，State 存進存檔 | 25 |
| 第 4 段 | ⭐ 接著跑：同一個 thread_id 接著跑、換一個 thread_id 從空白開始 | 25 |
| 第 5 段 | 一個 thread 的 State 歷程（每一步的存檔） | 25 |
| 第 6 段 | 選做：驗證怎麼做，改一句話看通過率 40% → 100% | 26–27 |

## 怎麼用

1. 先執行 **「0. 環境設定」**（這一格也會把 2-A 的手冊、2-B 的腳本、1-B 的工具準備好）。
2. 這本要改的只有 **thread_id**（第 3、4 段的表單）。其他表單想改也可以。
3. 每一段：**🤔 先寫預測 → ▶ 執行 → 👀 對照**；跑錯或斷線就從那一段的第一格重跑。

In [ ]:
#@title 0. 環境設定（第一次打開、或斷線後，先執行這一格）
#@markdown **模型來源**：`colab` ＝ Colab 內建模型（免費、不用 API key）；`gemini` ＝ 講師提供的 Gemini API key（備案）
模型來源 = "colab"  #@param ["colab", "gemini"]
#@markdown 選 `gemini` 時：建議在左邊 🔑「密鑰」新增 `GOOGLE_API_KEY`；也可以把 key 貼在下面（⚠️ 貼了就不要把 notebook 分享出去）
講師_API_KEY = ""  #@param {type:"string"}
#@markdown 進階（通常不用改）：指定模型名稱，留空用預設
colab_模型名稱 = ""  #@param {type:"string"}
gemini_模型名稱 = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. 用 uv 安裝套件（版本釘死，大約 10–30 秒）
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ 用 uv 安裝 LangGraph / LangChain ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4"', shell=True, check=True)

# 2. 寫出這本 notebook 用到的共用檔案
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_檔案 = {
    'llm.py': r'''"""模型轉接頭：把「Colab 內建模型」或「講師的 Gemini API key」包成同一個 model 物件。

學生不用看這個檔案。後面所有 cell 都只寫 model.invoke(...)、model.bind_tools(...)，
不管背後是哪一家模型，用法都一樣。

- colab ：google.colab.ai.generate_text（免費帳號可用，不需要 API key）
- gemini：langchain_google_genai.ChatGoogleGenerativeAI（需要 GOOGLE_API_KEY）

generate_text 只吃一段文字、只回一段文字，不支援原生的 tool calling。
所以 ColabAIChat 做了一個小轉接：
1. 把多則訊息接成一段文字（系統指示、使用者、AI、工具結果各標一個標題）
2. 有綁工具時，在文字最後附上工具清單，請模型「需要工具就只回一行 JSON」
3. 把模型回的 JSON 解析成 tool_calls，格式和原生 tool calling 一模一樣
這剛好讓你看到：模型從頭到尾只是在「說」它想用哪個工具。
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

設定檔 = Path(__file__).with_name(".llm.json")
預設_gemini_模型 = "gemini-flash-latest"


def 估算_token(text: str) -> int:
    """粗估 token 數：中日韓文字約 1 字 1 token，其他字元約 4 個 1 token。"""
    cjk = len(re.findall(r"[⺀-鿿豈-﫿＀-￯]", text))
    return max(1, round(cjk + (len(text) - cjk) / 4))


# ---------- 把訊息接成一段文字 ----------

def _工具說明(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params})：{desc}")
    return "\n".join(lines)


def _訊息轉文字(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"【系統指示】\n{m.text}")
        elif m.type == "human":
            parts.append(f"【使用者】\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("【AI】\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"【AI】\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "工具"
            parts.append(f"【工具 {name} 的結果】\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "這一次你一定要呼叫至少一個工具。\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"這一次你一定要呼叫 {tool_choice}。\n"
        parts.append(
            "【你可以用的工具】\n"
            f"{_工具說明(tools)}\n\n"
            "【回覆規則】\n"
            f"{must}"
            "如果你需要使用工具，只回一行 JSON，不要加任何其他文字、也不要加 ``` 符號：\n"
            '{"tool_calls": [{"name": "工具名稱", "args": {"參數名稱": "參數值"}}]}\n'
            "如果不需要工具（例如已經有足夠的資訊），就直接用一般文字回答，不要輸出 JSON。"
        )
    parts.append("【AI】")
    return "\n\n".join(parts)


def _解析回覆(text: str, tools: list[dict] | None) -> AIMessage:
    """模型回的是 JSON 紙條就轉成 tool_calls，否則當成一般文字。"""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """把 google.colab.ai.generate_text 包成 LangChain 的聊天模型。"""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _訊息轉文字(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _解析回覆(text, tools)
        message.usage_metadata = {
            "input_tokens": 估算_token(prompt),
            "output_tokens": 估算_token(text),
            "total_tokens": 估算_token(prompt) + 估算_token(text),
        }
        message.response_metadata["token_count"] = "估算值（Colab 內建模型不回報用量）"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- 對外：設定與取得 model ----------

def 讀設定() -> dict:
    try:
        return json.loads(設定檔.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def 存設定(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """記住選了哪個模型來源（不存 API key）。"""
    設定檔.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or 預設_gemini_模型},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _找_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = 讀設定()
    # 上雲端時沒有設定檔：用環境變數 WORKSHOP_PROVIDER=gemini（金鑰從 GOOGLE_API_KEY 讀）
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _找_api_key()
        if not key:
            raise RuntimeError(
                "選了 gemini 但找不到 API key：請在左邊 🔑「密鑰」新增 GOOGLE_API_KEY，"
                "或在「0. 環境設定」的表單貼上講師提供的 key，再重跑那一格。"
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or 預設_gemini_模型, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai 只能在 Google Colab 上使用；不在 Colab 的話請改選 gemini。") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""工作坊 notebook 共用的小工具（學生不用看）。

- 準備()          ：每一段的第一格都會呼叫，重新連上模型、重新載入改過的檔案
- 範例作業         ：表單下拉選單裡的幾段作業
- 記下() / 印出歷程()：把每次執行的 State 存起來，最後一格一次印出來給你截圖
- 顯示流程圖()      ：把 LangGraph 畫成圖
- 存到雲端硬碟() / 下載壓縮檔()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, 估算_token  # noqa: F401  （讓 notebook 可以直接 from workshop import 估算_token）

__all__ = [
    "BASE", "準備", "範例作業", "取得作業", "記下", "印出狀態", "印出訂單紙", "印出歷程", "清除歷程",
    "模型呼叫計數器", "顯示流程圖", "存到雲端硬碟", "下載壓縮檔", "估算_token",
]

BASE = Path(__file__).resolve().parent
歷程檔 = BASE / "history.jsonl"

範例作業 = {
    "少了右括號": "print('hi'\n",
    "正確的程式": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "空白": "",
    "變數打錯字": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "縮排亂掉": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "命名不一致": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def 取得作業(名稱: str) -> str:
    """表單選「我的作業」時讀 /content/review-bot/我的作業.py，其他從範例裡拿。"""
    if 名稱.startswith("我的作業"):
        path = BASE / "我的作業.py"
        if not path.exists():
            raise FileNotFoundError("還沒有「我的作業」：請先執行上面寫著 %%writefile ... 我的作業.py 的那一格。")
        return path.read_text(encoding="utf-8")
    return 範例作業[名稱]


def 準備():
    """每一段的起點：把 review-bot 放進 import 路徑、重新載入改過的檔案、回傳 model。"""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    return get_model()


# ---------- State 歷程 ----------

def _可存的(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _可存的(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_可存的(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def 記下(標題: str, 內容) -> None:
    """把這一次的結果（通常是 State）存進 history.jsonl。"""
    row = {"時間": _dt.datetime.now().strftime("%H:%M:%S"), "標題": 標題, "內容": _可存的(內容)}
    with 歷程檔.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _短(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"…（還有 {len(text) - n} 字）"


def 印出狀態(state: dict, 縮排: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{縮排}{k}: （{len(v)} 則訊息）")
            for m in v:
                m = _可存的(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{縮排}  [{m['type']}] 呼叫 {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{縮排}  [{m.get('type')}] {_短(m.get('content', ''), 200)}")
        else:
            print(f"{縮排}{k}: {_短(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def 印出訂單紙(第幾步: int, state: dict, 上一步: dict | None = None, 不印: tuple = ("code",)) -> None:
    """stream_mode="values" 每一步都會給整張訂單紙；順便標出這一步新寫了哪幾格。"""
    if not 上一步:
        說明 = "起點：空白訂單紙"
    else:
        新寫入 = [k for k in state if state.get(k) != 上一步.get(k)]
        說明 = "新寫入：" + ("、".join(新寫入) if 新寫入 else "（沒有改變）")
    print(f"第 {第幾步} 步（{說明}）")
    print("  ", {k: (_短(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in 不印})


def 印出歷程() -> None:
    if not 歷程檔.exists():
        print("還沒有任何紀錄：先跑上面幾段，再回來執行這一格。")
        return
    rows = [json.loads(line) for line in 歷程檔.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 共 {len(rows)} 筆紀錄（存在 {歷程檔}）\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['時間']}] {row['標題']} =====")
        內容 = row["內容"]
        if isinstance(內容, dict):
            印出狀態(內容)
        elif isinstance(內容, list):
            for j, item in enumerate(內容, 1):
                print(f"  --- 第 {j} 步 ---")
                if isinstance(item, dict):
                    印出狀態(item, "    ")
                else:
                    print(f"    {_短(item)}")
        else:
            print(f"  {_短(內容, 1000)}")
        print()


def 清除歷程() -> None:
    歷程檔.unlink(missing_ok=True)


# ---------- 數模型被呼叫了幾次 ----------

class 模型呼叫計數器(BaseCallbackHandler):
    """放進 config={"callbacks": [計數器]}，每呼叫一次模型就 +1。"""

    def __init__(self):
        self.次數 = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.次數 += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.次數 += 1


# ---------- 畫流程圖 ----------

def 顯示流程圖(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("（畫圖服務暫時連不上，改用文字版流程圖；可以貼到 https://mermaid.live 看圖）\n")
        print(g.draw_mermaid())


# ---------- 保存 ----------

def 存到雲端硬碟() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    目的地 = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, 目的地, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ 已存到雲端硬碟：{目的地}")
    print("   （notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」）")


def 下載壓縮檔() -> None:
    from google.colab import files

    名稱 = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(名稱), "zip", BASE.parent, BASE.name)
    print(f"✅ 已打包：{zip_path}")
    files.download(zip_path)
''',
    'tools/__init__.py': r'''"""作業小助教的工具箱：run_code、check_indent、read_spec、check_naming。"""
''',
    'tools/run_code.py': r'''"""run_code：在籠子裡執行學生的作業。

學生作業裡什麼都可能有（import os 刪檔、無限迴圈），所以這個工具有三道圍欄：
1. 執行前先檢查：不准 import os / sys / subprocess / shutil 這類危險模組，
   也不准用 eval、exec、open 和底線開頭的屬性（常被拿來繞過限制）
2. 開一個獨立的子程序執行，超過時間就強制停止（無限迴圈也停得下來）
3. 只回一行結果（第幾行、什麼錯），模型才看得懂

⚠️ 這是教學用的圍欄。正式產品請用真正的 sandbox（容器、RestrictedPython、e2b、Modal）。
"""

import ast
import json
import os
import subprocess
import sys
import tempfile

from langchain.tools import tool

禁止的模組 = {"os", "sys", "subprocess", "shutil", "socket", "pathlib", "ctypes", "importlib",
           "multiprocessing", "threading", "signal", "pickle", "builtins", "io", "inspect", "gc"}
允許的模組 = {"math", "random", "string", "re", "datetime", "collections", "itertools", "functools",
           "statistics", "time", "decimal", "fractions", "json", "copy", "heapq", "bisect"}
禁止的函式 = {"eval", "exec", "compile", "open", "input", "__import__", "globals", "locals", "vars",
           "getattr", "setattr", "delattr", "breakpoint", "help", "exit", "quit", "memoryview"}
禁止的屬性 = {"system", "popen", "modules", "builtins", "f_globals", "f_back", "gi_frame", "cr_frame"}
最長時間_秒 = 3
最長輸出_字 = 800

# 子程序裡真正執行作業的那段程式：用「縮小版的 builtins」執行，import 只放行白名單
_RUNNER = r"""
import builtins, contextlib, io, json, sys, traceback
code = sys.stdin.buffer.read().decode("utf-8")
allowed = set(json.loads(sys.argv[1]))
blocked = set(json.loads(sys.argv[2]))
real_import = builtins.__import__
def guarded_import(name, globals=None, locals=None, fromlist=(), level=0):
    if name.split(".")[0] not in allowed:
        raise ImportError("這個工具不允許 import " + name)
    return real_import(name, globals, locals, fromlist, level)
safe = {k: v for k, v in vars(builtins).items() if k not in blocked}
safe["__import__"] = guarded_import
try:
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (512 * 1024 * 1024, 512 * 1024 * 1024))
except Exception:
    pass
out = io.StringIO()
result = {"ok": True}
try:
    compiled = compile(code, "作業.py", "exec")
    with contextlib.redirect_stdout(out):
        exec(compiled, {"__builtins__": safe, "__name__": "__main__"})
except SyntaxError as e:
    result = {"ok": False, "line": e.lineno, "error": "SyntaxError: " + str(e.msg)}
except BaseException as e:
    line = None
    for frame in traceback.extract_tb(e.__traceback__):
        if frame.filename == "作業.py":
            line = frame.lineno
    result = {"ok": False, "line": line, "error": type(e).__name__ + ": " + str(e)}
result["stdout"] = out.getvalue()
sys.__stdout__.write(json.dumps(result))
"""


def _先檢查(code: str) -> str | None:
    """執行前的靜態檢查；有問題就回一行說明，沒問題回 None。"""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"❌ 第 {e.lineno} 行 SyntaxError: {e.msg}"
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names] if isinstance(node, ast.Import) else [node.module or ""]
            for name in names:
                root = name.split(".")[0]
                if root in 禁止的模組 or root not in 允許的模組:
                    return f"⛔ 已擋下：第 {node.lineno} 行 import {name}（這個工具不允許使用 {root}）"
        elif isinstance(node, ast.Name) and node.id in 禁止的函式:
            return f"⛔ 已擋下：第 {node.lineno} 行用了 {node.id}（這個工具不允許）"
        elif isinstance(node, ast.Attribute) and (
            node.attr.startswith("_") or node.attr in 禁止的屬性 or node.attr in 禁止的模組
        ):
            return f"⛔ 已擋下：第 {node.lineno} 行存取了 .{node.attr}（這個工具不允許）"
    return None


def 執行作業(code: str, 最長時間: float = 最長時間_秒) -> str:
    """真正做事的函式：檢查 → 子程序執行 → 回傳一行結果。"""
    if not code.strip():
        return "❌ 作業是空白的，沒有東西可以執行"
    擋下 = _先檢查(code)
    if 擋下:
        return 擋下
    env = {"PYTHONIOENCODING": "utf-8", "PATH": os.environ.get("PATH", "")}
    if os.name == "nt":
        env["SYSTEMROOT"] = os.environ.get("SYSTEMROOT", "")
    try:
        proc = subprocess.run(
            [sys.executable, "-I", "-c", _RUNNER, json.dumps(sorted(允許的模組)), json.dumps(sorted(禁止的函式))],
            input=code, capture_output=True, text=True, encoding="utf-8",
            timeout=最長時間, cwd=tempfile.gettempdir(), env=env,
        )
    except subprocess.TimeoutExpired:
        return f"⏱️ 執行超過 {最長時間:g} 秒被強制停止（可能有無限迴圈）"
    try:
        result = json.loads(proc.stdout.strip().splitlines()[-1])
    except (IndexError, json.JSONDecodeError):
        return "❌ 執行失敗（程式沒有正常結束，可能用了太多記憶體）"
    stdout = result.get("stdout", "")
    if len(stdout) > 最長輸出_字:
        stdout = stdout[:最長輸出_字] + "\n…（輸出太長，後面省略）"
    if result["ok"]:
        return f"✅ 執行成功，輸出：\n{stdout}" if stdout else "✅ 執行成功（沒有印出任何東西）"
    where = f"第 {result['line']} 行 " if result.get("line") else ""
    印出 = f"\n（出錯前印出：{stdout.strip()}）" if stdout.strip() else ""
    return f"❌ {where}{result['error']}{印出}"


@tool
def run_code(code: str) -> str:
    """安全地執行一段 Python 作業，回傳輸出或錯誤訊息。

    什麼時候用：想知道作業「實際跑起來」會印出什麼、會不會出錯時。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：一行結果，例如「✅ 執行成功，輸出：5」或「❌ 第 3 行 NameError: ...」。
    限制：不能 import os、sys、subprocess 等模組；超過 3 秒會被停止。
    """
    return 執行作業(code)
''',
    'tools/read_spec.py': r'''"""read_spec：讀這一題的作業要求。

題目要求存在 review-bot/spec.md，notebook 的表單改了題目就會重寫這個檔案。
"""

from pathlib import Path

from langchain.tools import tool

題目檔 = Path(__file__).resolve().parent.parent / "spec.md"


def 讀題目() -> str:
    if not 題目檔.exists():
        return "（這一題沒有提供作業要求）"
    return 題目檔.read_text(encoding="utf-8").strip() or "（這一題沒有提供作業要求）"


@tool
def read_spec() -> str:
    """讀取這一題的作業要求（題目寫了什麼、要交什麼）。

    什麼時候用：要判斷作業有沒有做到題目要求之前，例如函式名稱、輸出格式對不對。
    不需要參數。
    回傳：題目要求的全文。
    """
    return 讀題目()
''',
    'tools/check_naming.py': r'''"""check_naming：把 Skill 的腳本 scripts/check_naming.py 包成 agent 的工具。

腳本外面包一層 @tool，就進了 agent 的工具箱：
AI 只負責「決定要不要跑」和「把結果抄進回饋」，檢查本身是 Python，每次都一樣。
"""

import subprocess
import sys
from pathlib import Path

from langchain.tools import tool

腳本 = Path(__file__).resolve().parent.parent / "skills" / "naming-check" / "scripts" / "check_naming.py"


def 執行命名腳本(code: str) -> str:
    proc = subprocess.run(
        [sys.executable, str(腳本)], input=code.encode("utf-8"), capture_output=True, timeout=10
    )
    out = proc.stdout.decode("utf-8").strip()
    return out or "腳本執行失敗：" + proc.stderr.decode("utf-8", "replace").strip()[-200:]


@tool
def check_naming(code: str) -> str:
    """檢查變數與函式名稱是否為小寫加底線（snake_case），回傳不合規則的行。

    什麼時候用：批改作業、需要列出命名問題時（這是 naming-check 手冊附的腳本）。
    參數 code：完整的 Python 程式碼（字串）。
    回傳：每個問題一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」；全部合格時回傳「OK：…」。
    """
    return 執行命名腳本(code)
''',
    'graph.py': r'''"""1-A：作業小助教的 workflow，路線由程式決定。

START → check_syntax ─語法 OK→ give_feedback → check_format ─格式 OK→ END
                     └語法錯→ explain_error → END     └不合格、retries 還沒到上限→ 退回 give_feedback

三個零件：
- State：一張訂單紙（dict），從頭傳到尾
- Node ：一個工作站（函式），收 State、只回傳自己改的那幾格
- Edge ：工作站之間的箭頭；條件邊就是一個回傳「下一站名字」的 if
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # 學生交的作業
    syntax_ok: bool | None # 語法對不對（還沒檢查時是 None）
    error: str             # 語法錯在哪裡
    feedback: str          # 給學生的回饋
    format_ok: bool | None # 回饋格式合不合格
    format_problem: str    # 不合格的原因（退回重寫時會告訴 AI）
    retries: int           # 回饋寫了幾次


# ---------- Node 1：check_syntax（純 Python） ----------

_錯誤翻譯 = [
    (r"'(.)' was never closed", lambda m: f"少了右括號 {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "字串少了結尾的引號"),
    (r"expected ':'", lambda m: "少了冒號 :"),
    (r"unexpected indent", lambda m: "縮排多了"),
    (r"expected an indented block", lambda m: "這裡需要縮排"),
    (r"unindent does not match", lambda m: "縮排對不齊"),
    (r"unmatched '(.)'", lambda m: f"多了一個 {m.group(1)}"),
    (r"invalid character", lambda m: "有全形或看不懂的字元"),
    (r"Perhaps you forgot a comma", lambda m: "可能少了逗號"),
    (r"invalid syntax", lambda m: "語法不正確"),
]


def 翻譯語法錯誤(msg: str) -> str:
    for pattern, 說法 in _錯誤翻譯:
        m = re.search(pattern, msg)
        if m:
            return 說法(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "作業是空白的，沒有任何程式"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"第 {e.lineno} 行{翻譯語法錯誤(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- 條件邊 1：語法對 → 給回饋；語法錯 → 解釋錯誤 ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2：explain_error（純 Python） ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"語法：{state['error']}\n改進：先把這一行修好，再執行一次看看。"}


# ---------- Node 3：give_feedback（兩個版本） ----------

def give_feedback_固定文字(state: State) -> dict:
    """先不接 AI 的版本：回固定的一段話。"""
    return {
        "feedback": "優點：語法正確，程式可以執行。\n改進：（這一版還沒接 AI，先用固定文字）",
        "retries": state.get("retries", 0) + 1,
    }


def 建立_give_feedback(model, 回饋字數上限: int):
    """接上 AI 的版本：AI 只是住在這個節點裡的一行程式。"""

    def give_feedback(state: State) -> dict:
        prompt = (
            "你是 Python 作業小助教，請用繁體中文給下面這段作業回饋。\n"
            "格式規定（一定要遵守）：\n"
            "- 只寫兩段：第一段以「優點：」開頭，第二段以「改進：」開頭\n"
            f"- 兩段加起來不超過 {回饋字數上限} 個字\n\n"
            f"作業：\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\n你上一版的回饋被退回，原因：{state['format_problem']}。請修正後重寫。"
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4：check_format（純 Python 的檢查站） ----------

def 建立_check_format(回饋字數上限: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        問題 = []
        if "優點" not in feedback:
            問題.append("少了「優點」段落")
        if "改進" not in feedback:
            問題.append("少了「改進」段落")
        if len(feedback) > 回饋字數上限:
            問題.append(f"太長了：{len(feedback)} 字，上限 {回饋字數上限} 字")
        return {"format_ok": not 問題, "format_problem": "；".join(問題)}

    return check_format


# ---------- 條件邊 2：格式 OK 或重寫次數到了 → 結束；否則退回重寫 ----------

def 建立_route_after_format(最多重試次數: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= 最多重試次數:
            return END  # 重寫次數到了就停，不然會無限轉圈
        return "give_feedback"

    return route_after_format


# ---------- 把零件接起來 ----------

def 建立小助教(model=None, 最多重試次數: int = 3, 回饋字數上限: int = 150):
    """model=None 時 give_feedback 用固定文字（先不接 AI）。"""
    give_feedback = 建立_give_feedback(model, 回饋字數上限) if model else give_feedback_固定文字

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", 建立_check_format(回饋字數上限))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", 建立_route_after_format(最多重試次數), ["give_feedback", END])
    return builder.compile()


def 空白訂單紙(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}
''',
    'skill_loader.py': r'''"""skill_loader：AI 選手冊只看「名片」，選中了才翻開內文。

漸進式揭露三層：
1. 書背（名片）：SKILL.md 開頭兩條 --- 之間的 name 和 description，約 100 token，一直在桌上
2. 內文：整份 SKILL.md，被選中才讀
3. 附錄：scripts/、references/ 裡的檔案，用到才打開
"""

from pathlib import Path

SKILLS = Path(__file__).resolve().parent / "skills"


def 讀名片(path: Path) -> dict:
    """只讀兩條 --- 之間的 name 和 description。"""
    text = Path(path).read_text(encoding="utf-8")
    card = {"name": Path(path).parent.name, "description": "", "path": str(path)}
    if text.startswith("---"):
        head = text.split("---", 2)[1]
        for line in head.strip().splitlines():
            key, _, value = line.partition(":")
            if key.strip() in ("name", "description"):
                card[key.strip()] = value.strip()
    return card


def 讀內文(card: dict) -> str:
    """選中之後才翻開：名片以下的手冊內容。"""
    text = Path(card["path"]).read_text(encoding="utf-8")
    return text.split("---", 2)[2].strip() if text.startswith("---") else text


def 列出名片(只要: list[str] | None = None) -> list[dict]:
    cards = [讀名片(p) for p in sorted(SKILLS.glob("*/SKILL.md"))]
    if 只要:
        cards = [c for c in cards if Path(c["path"]).parent.name in 只要]
    return cards


def 挑手冊(model, 需求: str, cards: list[dict]) -> dict:
    """把所有名片給模型看，讓它挑一本；回傳被挑中的那張名片。"""
    清單 = "\n".join(f"- {c['name']}：{c['description']}" for c in cards)
    回答 = model.invoke(
        "你要替下面的批改需求挑一本最適合的手冊。\n"
        f"批改需求：{需求}\n\n"
        f"手冊名片：\n{清單}\n\n"
        "只回答手冊的 name，不要其他字。"
    ).text.strip()
    提到的 = [(回答.find(c["name"]), c) for c in cards if c["name"] in 回答]
    return min(提到的, key=lambda x: x[0])[1] if 提到的 else cards[0]
''',
    'spec.md': r'''請寫一個函式 add(a, b)，回傳兩個數字相加的結果，並用 print(add(2, 3)) 印出 5。
''',
    'skills/friendly-feedback/SKILL.md': r'''---
name: friendly-feedback
description: 給 Python 初學者的作業回饋。語氣溫和，先講優點再講改進。適合第一次交作業、剛學會 Python 的同學。
---

# 溫和回饋規則

1. 先講一個做得好的地方
2. 最多指出兩個問題，每個都附上改法
3. 不使用「錯誤」「很差」這類字眼
4. 回饋分兩段：第一段以「優點：」開頭，第二段以「改進：」開頭
''',
    'skills/strict-review/SKILL.md': r'''---
name: strict-review
description: 期中考、正式評分用的嚴格審查。列出所有問題、不留情面、附上扣分。
---

# 嚴格審查規則

1. 列出所有問題，一個都不放過，每個問題都標出第幾行
2. 不說客套話，不用「很棒」「不錯」這類字眼
3. 每個問題附上扣幾分（滿分 100 分）
4. 回饋分兩段：第一段以「優點：」開頭（真的沒有就寫「優點：無」），第二段以「改進：」開頭
''',
    'skills/naming-check/SKILL.md': r'''---
name: naming-check
description: 檢查 Python 作業的變數與函式名稱是不是小寫英文加底線（snake_case）。批改作業、需要列出命名問題時使用。
---

# 命名檢查

執行 scripts/check_naming.py（工具名稱 check_naming），把它的輸出寫進回饋。
不要自己判斷、不要增加或刪掉任何一項，照抄它的每一行。

輸出格式：每個問題一行，例如「第 3 行：myVar」
''',
    'skills/naming-check/scripts/check_naming.py': r'''"""檢查變數與函式名稱是不是小寫英文加底線（snake_case）。

用法：
    python check_naming.py 作業.py        # 檢查一個檔案
    python check_naming.py < 作業.py      # 或從標準輸入讀
輸出：
    每個問題一行，例如「第 3 行：myVar 不符合規則（建議 my_var）」
    全部合格時印「OK：所有變數與函式名稱都是小寫加底線」

一個腳本只做一件事、輸出短短幾行，AI 才看得懂；每次執行結果都一模一樣。
"""

import ast
import re
import sys

規則 = re.compile(r"^_?[a-z][a-z0-9_]*$")


def 建議名稱(name: str) -> str:
    s = re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", name)
    return re.sub(r"_+", "_", s).lower()


def 找出所有名稱(tree: ast.AST) -> dict[str, int]:
    """名稱 → 第一次出現的行號（變數、函式、參數，不含 class 名稱）。"""
    名稱 = {}

    def 記(name, line):
        if name not in 名稱 or line < 名稱[name]:
            名稱[name] = line

    for node in ast.walk(tree):
        if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Store):
            記(node.id, node.lineno)
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            記(node.name, node.lineno)
        elif isinstance(node, ast.arg) and node.arg not in ("self", "cls"):
            記(node.arg, node.lineno)
    return 名稱


def 檢查命名(code: str) -> str:
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return f"無法檢查：第 {e.lineno} 行有語法錯誤"
    問題 = [
        f"第 {line} 行：{name} 不符合規則（建議 {建議名稱(name)}）"
        for name, line in sorted(找出所有名稱(tree).items(), key=lambda x: (x[1], x[0]))
        if not 規則.match(name)
    ]
    return "\n".join(問題) if 問題 else "OK：所有變數與函式名稱都是小寫加底線"


if __name__ == "__main__":
    if len(sys.argv) > 1:
        with open(sys.argv[1], encoding="utf-8") as f:
            source = f.read()
    else:
        source = sys.stdin.buffer.read().decode("utf-8")
    sys.stdout.reconfigure(encoding="utf-8")
    print(檢查命名(source))
''',
}
for rel, content in _檔案.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))

# 3. 記住模型來源（不會存 API key），測試連線
import llm
llm.存設定(模型來源, colab_模型名稱.strip(), gemini_模型名稱.strip())
if 講師_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = 講師_API_KEY.strip()
from workshop import *
model = 準備()

回覆 = model.invoke("用一句話跟剛學 Python 的同學打招呼").text
print(f"✅ 環境設定完成，模型來源：{模型來源}")
print(f"🤖 模型說：{回覆.strip()}")

---
## 第 1 段：雙核長什麼樣

> 📍 **對應簡報**：投影片 24「雙核：把上週的流程和今天的手冊接起來」、投影片 25「兩個新零件」

下一格寫出 `dual_core.py`（不用改，直接執行）。想看程式的話，重點在三個地方：

- `建立雙核小助教`：外層的節點和箭頭，跟 1-A 幾乎一樣，只多了 `skill_loader` 這一站
- `建立_skill_loader`：把所有手冊的**名片**給模型看，挑一本，內文放進 State 的 `skill`
- `建立_give_feedback`：裡面用 `create_agent` 做一個小 agent，手冊當 system prompt，工具箱有 `check_naming`、`run_code`、`read_spec`

### 🤔 預測題
1. 外層流程圖上，`skill_loader` 會放在哪兩站中間？
2. 流程圖上看得到裡面那個 agent 會叫哪些工具嗎？

In [ ]:
%%writefile /content/review-bot/dual_core.py
"""2-D：雙核作業小助教。外面是 1-A 的 workflow（順序鎖死），裡面是 1-B 的 agent（放手判斷）。

START → check_syntax ─語法 OK→ skill_loader → give_feedback → check_format ─OK→ END
                     └語法錯→ explain_error → END            └不合格→ 退回 give_feedback

- skill_loader ：看所有手冊的名片，挑一本，把內文放進 State（2-A 的手冊）
- give_feedback：裡面是一個小 agent，用挑中的手冊當 system prompt，
                 工具箱有 run_code、read_spec、check_naming（2-B 的腳本包成工具）
- checkpointer ：每一步的 State 都存進 SQLite 檔，同一個 thread_id 可以接著跑
"""

import operator
import sqlite3
from pathlib import Path
from typing import Annotated, TypedDict

from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph

from graph import check_syntax, explain_error, route_after_syntax, 建立_check_format, 建立_route_after_format
from skill_loader import 列出名片, 挑手冊, 讀內文
from tools.check_naming import check_naming
from tools.read_spec import read_spec
from tools.run_code import run_code

存檔位置 = Path(__file__).resolve().parent / "checkpoints.sqlite"
可選的手冊 = ["friendly-feedback", "strict-review"]


class 雙核State(TypedDict, total=False):
    code: str                 # 學生交的作業
    request: str              # 批改需求（skill_loader 用它挑手冊）
    syntax_ok: bool | None
    error: str
    skill_name: str           # 挑中的手冊
    skill: str                # 手冊內文
    feedback: str
    format_ok: bool | None
    format_problem: str
    retries: int
    tools_used: Annotated[list[str], operator.add]  # 裡面的 agent 用過哪些工具（往下接著寫）


def 建立_skill_loader(model):
    def skill_loader(state: 雙核State) -> dict:
        card = 挑手冊(model, state.get("request", ""), 列出名片(可選的手冊))
        return {"skill_name": card["name"], "skill": 讀內文(card)}

    return skill_loader


def 建立_give_feedback(model, 回饋字數上限: int):
    def give_feedback(state: 雙核State) -> dict:
        system_prompt = (
            f"{state['skill']}\n\n"
            "# 你的工具\n"
            "- check_naming：檢查變數命名（手冊附的腳本），要談命名時用它，照抄它的結果\n"
            "- run_code：執行作業看輸出\n"
            "- read_spec：讀題目要求\n"
            "要不要用、用哪一個，你自己決定。\n\n"
            "# 格式（外層流程會用程式檢查，不合格會退回）\n"
            "- 只寫兩段：第一段以「優點：」開頭，第二段以「改進：」開頭\n"
            f"- 兩段加起來不超過 {回饋字數上限} 個字"
        )
        if state.get("format_problem"):
            system_prompt += f"\n\n上一版被退回，原因：{state['format_problem']}。請修正。"
        小agent = create_agent(model, tools=[check_naming, run_code, read_spec], system_prompt=system_prompt)
        result = 小agent.invoke(
            {"messages": [{"role": "user", "content": f"請批改這份作業：\n```python\n{state['code']}\n```"}]},
            config={"recursion_limit": 25},
        )
        messages = result["messages"]
        used = [c["name"] for m in messages for c in (getattr(m, "tool_calls", None) or [])]
        return {"feedback": messages[-1].text.strip(), "retries": state.get("retries", 0) + 1, "tools_used": used}

    return give_feedback


def 逐站執行(小助教, 輸入, config: dict) -> dict:
    """一站一站跑，印出每一站寫了哪幾格；回傳跑完（或中斷）時存檔裡的 State。"""
    for event in 小助教.stream(輸入, config=config, stream_mode="updates"):
        for 站, 寫入 in event.items():
            if 站 == "__interrupt__":
                print("⏸️  中斷：流程停在這裡，State 已經存進存檔")
                continue
            print(f"▶ {站}")
            for key, value in (寫入 or {}).items():
                text = value if isinstance(value, str) else repr(value)
                if key == "skill":
                    text = text.splitlines()[0] + " …（手冊內文）"
                print(f"    {key}: {text[:200]}")
    return 小助教.get_state(config).values


def 開存檔(path: Path = 存檔位置) -> SqliteSaver:
    """把 State 存進 SQLite 檔：重跑 cell、甚至重新啟動執行階段，存檔都還在。"""
    return SqliteSaver(sqlite3.connect(str(path), check_same_thread=False))


def 建立雙核小助教(model, checkpointer=None, 最多重試次數: int = 3, 回饋字數上限: int = 250,
                interrupt_before: list[str] | None = None):
    builder = StateGraph(雙核State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("skill_loader", 建立_skill_loader(model))
    builder.add_node("give_feedback", 建立_give_feedback(model, 回饋字數上限))
    builder.add_node("check_format", 建立_check_format(回饋字數上限))

    builder.add_edge(START, "check_syntax")
    # 語法 OK 先去挑手冊，再給回饋
    builder.add_conditional_edges(
        "check_syntax", route_after_syntax, {"give_feedback": "skill_loader", "explain_error": "explain_error"}
    )
    builder.add_edge("explain_error", END)
    builder.add_edge("skill_loader", "give_feedback")
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", 建立_route_after_format(最多重試次數), ["give_feedback", END])
    return builder.compile(checkpointer=checkpointer, interrupt_before=interrupt_before)


In [ ]:
#@title ▶ 執行：畫出雙核的外層流程圖
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import dual_core

小助教 = dual_core.建立雙核小助教(model)
顯示流程圖(小助教)

### 👀 你看到了什麼

- `skill_loader` 在 `check_syntax`（語法 OK）和 `give_feedback` 中間：先檢查語法、再挑手冊、再給回饋、再檢查格式。**這個順序是程式鎖死的。**
- 流程圖上看不到 agent 會叫哪些工具，因為那是 `give_feedback` 這一站**裡面**的事，由 AI 當下決定。
  外面鎖住不能錯的事，裡面放手讓它判斷：這就是雙拼。

---
## 第 2 段：完整跑一次

> 📍 **對應簡報**：投影片 24「雙核」講師 demo

### 🤔 預測題
1. 作業選「命名不一致」、需求選「初學者」，`skill_loader` 會挑哪一本手冊？換成「期中考」呢？
2. 裡面的 agent 會用 `check_naming` 嗎？會不會也用 `run_code`？
3. 如果它寫的回饋太長，誰會把它退回去？

In [ ]:
#@title ▶ 執行：雙核小助教，一站一站跑
作業 = "命名不一致"  #@param ["命名不一致", "變數打錯字", "正確的程式", "少了右括號"]
批改需求 = "幫第一次交作業的初學者看看"  #@param ["幫第一次交作業的初學者看看", "期中考正式評分，請嚴格審查"] {allow-input: true}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import time
import dual_core

小助教 = dual_core.建立雙核小助教(model, checkpointer=dual_core.開存檔())
config = {"configurable": {"thread_id": f"完整示範-{time.strftime('%H%M%S')}"}}
state = dual_core.逐站執行(小助教, {"code": 取得作業(作業), "request": 批改需求, "retries": 0}, config)

print("\n========== 結果 ==========")
print("挑中的手冊：", state.get("skill_name"))
print("裡面的 agent 用了哪些工具：", state.get("tools_used") or "（沒有用工具）")
print(f"retries = {state.get('retries')}，format_ok = {state.get('format_ok')}")
print("\n回饋：")
print(state.get("feedback"))
記下(f"2-D 第 2 段：完整跑一次（{作業}／{批改需求}）", {k: v for k, v in state.items() if k != "skill"})

### 👀 你看到了什麼

- 外層每次都照同樣的順序走：`check_syntax → skill_loader → give_feedback → check_format`。換作業、換需求都一樣。
- `skill_loader` 只看名片挑手冊：需求寫「初學者」大多挑 `friendly-feedback`，寫「期中考、嚴格」大多挑 `strict-review`。
- 裡面的 agent 用了哪些工具、用幾次，**每次可能不同**；但它最後交出的回饋一定會被 `check_format` 用程式檢查，不合格就退回重寫。
- 試著把「少了右括號」丟進去：根本不會走到 agent，`check_syntax` 之後就直接去 `explain_error`，一個 token 都不花。

---
## 第 3 段：中斷，跑到一半先停

> 📍 **對應簡報**：投影片 25「還有一個叫 checkpointer 的東西能把 State 存起來、中斷後接著跑」

**checkpointer** 會在每一站跑完時把整張訂單紙（State）存起來；**thread_id** 是這份存檔的編號，像置物櫃的號碼。
這裡的存檔放在 `/content/review-bot/checkpoints.sqlite` 這個檔案裡，所以重跑 cell、甚至重新啟動執行階段，存檔都還在。

下面這格會在 `give_feedback` **前面**故意停下來（模擬跑到一半斷線）。

### 🤔 預測題
停下來的時候，存檔裡的 State 會有哪幾格已經填好？`feedback` 有內容嗎？

In [ ]:
#@title ▶ 執行：跑到 give_feedback 前面就先停
thread_id = "小明的作業"  #@param {type:"string"}
作業 = "命名不一致"  #@param ["命名不一致", "變數打錯字", "正確的程式", "少了右括號"]
批改需求 = "幫第一次交作業的初學者看看"  #@param ["幫第一次交作業的初學者看看", "期中考正式評分，請嚴格審查"] {allow-input: true}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import dual_core

小助教 = dual_core.建立雙核小助教(model, checkpointer=dual_core.開存檔(), interrupt_before=["give_feedback"])
config = {"configurable": {"thread_id": thread_id}}
if 小助教.get_state(config).values:
    print(f"（提醒：thread_id「{thread_id}」本來就有存檔了，這次會在同一份存檔上再跑一次）\n")

state = dual_core.逐站執行(小助教, {"code": 取得作業(作業), "request": 批改需求, "retries": 0}, config)
存檔 = 小助教.get_state(config)
print(f"\n📂 thread_id「{thread_id}」的存檔：")
印出狀態({k: v for k, v in 存檔.values.items() if k not in ("skill", "code")})
print("下一站（還沒跑）：", 存檔.next)
記下(f"2-D 第 3 段：中斷（thread_id={thread_id}）", {"next": list(存檔.next), **{k: v for k, v in 存檔.values.items() if k != "skill"}})

### 👀 你看到了什麼

- 停在 `give_feedback` 前面：`syntax_ok`、`skill_name` 已經填好，`feedback` 還是空的。
- 「下一站」是 `('give_feedback',)`：存檔不只記得 State，也記得**流程走到哪裡**。

---
## 第 4 段 ⭐：接著跑，還是從空白開始？

> 📍 **對應簡報**：投影片 25；第 2 堂最低完成標準「2-D 換 thread_id 重跑並說明 State 有沒有保留」

下面這格會先用 thread_id 去存檔裡找：找到「停在一半」的存檔就**接著跑**；找不到就**從空白開始**。

### 🤔 預測題（兩輪，答案寫在紙上）
1. thread_id 保持 **小明的作業**（跟第 3 段一樣），執行。它會從哪一站開始？`skill_name` 需要重挑嗎？
2. 把 thread_id 改成 **小華的作業**，再執行一次。這次 State 裡有什麼？

In [ ]:
#@title ▶ 執行：用 thread_id 找存檔，接著跑或從空白開始
thread_id = "小明的作業"  #@param {type:"string"}
作業 = "命名不一致"  #@param ["命名不一致", "變數打錯字", "正確的程式", "少了右括號"]
批改需求 = "幫第一次交作業的初學者看看"  #@param ["幫第一次交作業的初學者看看", "期中考正式評分，請嚴格審查"] {allow-input: true}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import dual_core

小助教 = dual_core.建立雙核小助教(model, checkpointer=dual_core.開存檔())
config = {"configurable": {"thread_id": thread_id}}
存檔 = 小助教.get_state(config)

if not 存檔.values:
    print(f"🆕 thread_id「{thread_id}」沒有任何存檔：State 是空白的，從頭開始跑\n")
    state = dual_core.逐站執行(小助教, {"code": 取得作業(作業), "request": 批改需求, "retries": 0}, config)
    狀況 = "沒有存檔，從空白開始"
elif 存檔.next:
    print(f"📂 找到 thread_id「{thread_id}」的存檔：停在 {存檔.next} 前面，接著跑")
    print("   存檔裡已經有：", {k: v for k, v in 存檔.values.items() if k in ("syntax_ok", "skill_name", "retries")}, "\n")
    state = dual_core.逐站執行(小助教, None, config)      # 輸入 None ＝ 從存檔接著跑
    狀況 = "找到存檔，接著跑"
else:
    print(f"✅ thread_id「{thread_id}」已經跑完了，存檔內容如下（想重來就換一個 thread_id）\n")
    state = 存檔.values
    狀況 = "已經跑完"

print("\n========== 現在的 State ==========")
印出狀態({k: v for k, v in state.items() if k not in ("skill", "code")})
記下(f"2-D 第 4 段：{狀況}（thread_id={thread_id}）", {k: v for k, v in state.items() if k != "skill"})

### 👀 你看到了什麼

- **同一個 thread_id**：直接從 `give_feedback` 接著跑，`check_syntax` 和 `skill_loader` 沒有重跑，`skill_name` 是存檔裡原本那一本。**State 沒有丟失。**
- **換一個 thread_id**：存檔裡什麼都沒有，State 從空白開始，整條流程從 `check_syntax` 重新跑。
- 同一個 thread_id 跑完之後再執行一次，會看到「已經跑完」：存檔一直都在，除非你換編號。
- 這就是原提案說的「受控節點正確觸發 Skill，且 State 未丟失」。正式上線時通常把 SQLite 換成資料庫（例如 Postgres），做法一樣。

---
## 第 5 段：一個 thread 的 State 歷程

> 📍 **對應簡報**：投影片 25（checkpointer）

checkpointer 不只存「最新的」State，而是**每一站跑完都存一份**。下面把同一個 thread_id 的每一份存檔列出來。

### 🤔 預測題
「小明的作業」跑完整條流程，總共會有幾份存檔？

In [ ]:
#@title ▶ 執行：列出一個 thread_id 的每一份存檔
thread_id = "小明的作業"  #@param {type:"string"}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
import dual_core

小助教 = dual_core.建立雙核小助教(model, checkpointer=dual_core.開存檔())
config = {"configurable": {"thread_id": thread_id}}
歷史 = list(小助教.get_state_history(config))
if not 歷史:
    print(f"thread_id「{thread_id}」沒有任何存檔。先跑第 3、4 段。")
for snap in reversed(歷史):
    v = snap.values
    步 = snap.metadata.get("step")
    print(f"{'收到輸入' if 步 == -1 else f'第 {步} 步'}｜下一站：{snap.next or '（結束）'}｜"
          f"syntax_ok={v.get('syntax_ok')}  skill={v.get('skill_name')}  "
          f"retries={v.get('retries')}  format_ok={v.get('format_ok')}  有回饋={'是' if v.get('feedback') else '否'}")
記下(f"2-D 第 5 段：State 歷程（thread_id={thread_id}）",
     [{"step": s.metadata.get("step"), "next": list(s.next),
       **{k: val for k, val in s.values.items() if k not in ("skill", "code")}} for s in reversed(歷史)])

### 👀 你看到了什麼

- 每跑完一站就多一份存檔，所以可以看到訂單紙一格一格被填滿的過程，跟 1-A 的 `stream_mode="values"` 很像，只是這次是**存在檔案裡**的。
- 這就是 harness 裡的「Replay」：失敗的時候，可以回到中間某一步重新試，不用每次從頭跑。

---
## 第 6 段（選做）：驗證怎麼做？把「應該怎麼做」寫成程式能判斷的規則

> 📍 **對應簡報**：投影片 26「驗證怎麼做」、投影片 27「改一句話，量化看到變好：40% → 100%」

把「agent 應該怎麼做」寫成五個考題（eval case）：**第一個該叫哪個工具**、**最後的回覆要有哪些字**。
考卷是程式在改，不是人在看。我們比較兩個版本的 system prompt：

- **baseline**：沒有特別交代順序
- **v2**：多一句「一定要先呼叫 check_syntax 再回饋」

⚠️ 這一段會呼叫模型大約 20–30 次，免費額度不多的話**課後再跑**：把表單的「要跑驗證嗎」打勾再執行。

### 🤔 預測題
baseline 五題會過幾題？加了一句之後呢？會不會剛好 100%？

In [ ]:
#@title ▶ 執行（選做）：五個考題，比較 baseline 和 v2 的通過率
要跑驗證嗎 = False  #@param {type:"boolean"}
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
from langchain.agents import create_agent
from langchain.tools import tool
import graph
from tools.read_spec import read_spec
from tools.run_code import run_code

@tool
def check_syntax(code: str) -> str:
    """檢查 Python 作業的語法對不對。回傳「語法 OK」或錯在第幾行。"""
    結果 = graph.check_syntax({"code": code})
    return "語法 OK" if 結果["syntax_ok"] else f"語法錯誤：{結果['error']}"

# 五個考題：第一個該叫誰、最後要有什麼
EVAL_CASES = [
    {"id": "001_must_check_syntax_first", "作業": "正確的程式", "must_call_first": "check_syntax", "must_contain_in_final": ["優點", "改進"]},
    {"id": "002_handle_syntax_error", "作業": "少了右括號", "must_call_first": "check_syntax", "must_contain_in_final": ["語法"]},
    {"id": "003_wrong_variable", "作業": "變數打錯字", "must_call_first": "check_syntax", "must_contain_in_final": ["改進"]},
    {"id": "004_messy_indent", "作業": "縮排亂掉", "must_call_first": "check_syntax", "must_contain_in_final": ["改進"]},
    {"id": "005_naming", "作業": "命名不一致", "must_call_first": "check_syntax", "must_contain_in_final": ["優點", "改進"]},
]

def evaluate_one(考生, case):
    inputs = {"messages": [{"role": "user", "content": f"幫我批改這份作業：\n```python\n{取得作業(case['作業'])}\n```"}]}
    events = list(考生.stream(inputs, stream_mode="updates"))
    msgs = [m for e in events for u in e.values() for m in (u or {}).get("messages", [])]
    tc = [c["name"] for m in msgs for c in (getattr(m, "tool_calls", None) or [])]   # 工具呼叫順序
    final = next((m.text for m in reversed(msgs) if m.type == "ai" and not m.tool_calls), "")
    reasons = []
    if not tc:
        reasons.append(f"沒呼叫任何 tool（預期 {case['must_call_first']}）")
    elif tc[0] != case["must_call_first"]:
        reasons.append(f"第一個 tool 是 {tc[0]}，預期 {case['must_call_first']}")
    for kw in case["must_contain_in_final"]:
        if kw not in final:
            reasons.append(f"最終回覆缺關鍵字 {kw!r}")
    return {"id": case["id"], "passed": not reasons, "reasons": reasons, "tool_calls": tc}

基本 = "你是 Python 作業小助教，用繁體中文給初學者回饋。回饋分兩段：第一段以「優點：」開頭，第二段以「改進：」開頭。"
版本們 = {
    "baseline（沒交代順序）": 基本,
    "v2（加一句「先呼叫 check_syntax 再回饋」）": 基本 + "一定要先呼叫 check_syntax 檢查語法，再回饋。",
}

if not 要跑驗證嗎:
    print("這一段是選做：把上面表單的「要跑驗證嗎」打勾，再執行一次。")
else:
    成績 = {}
    for 版本, system_prompt in 版本們.items():
        考生 = create_agent(model, tools=[check_syntax, run_code, read_spec], system_prompt=system_prompt)
        print(f"===== {版本} =====")
        結果們 = [evaluate_one(考生, case) for case in EVAL_CASES]
        for r in 結果們:
            print(f"[{'PASS' if r['passed'] else 'FAIL'}] {r['id']}  tool_calls={r['tool_calls']}")
            for why in r["reasons"]:
                print(f"        - {why}")
        通過 = sum(r["passed"] for r in 結果們)
        print(f"通過 {通過}/{len(結果們)} ({通過 / len(結果們):.0%})\n")
        成績[版本] = f"{通過}/{len(結果們)}"
    記下("2-D 第 6 段：eval 通過率", 成績)

### 👀 你看到了什麼

- baseline 通常有幾題的第一個工具不是 `check_syntax`：沒交代，它就照自己的想法走。
- 加一句之後通過率明顯上升，但**不一定永遠 100%**：模型偶爾還是會不聽話。有客觀的分數可以比，才知道改了是不是真的變好。
- 再往前一步：把「第一個一定要 check_syntax」直接寫成**外層流程的順序**、把「一定要有優點和改進」寫成 `check_format` 節點，
  就不用事後考、也不用花 token。這正是第 1、2 段的雙核做的事（也是 2-B 放法 C 的論點）。

## 收尾：存檔與交作業

- 💾 **存到雲端硬碟**：把 `/content/review-bot/`（你改過的手冊、程式、State 紀錄）存一份到自己的 Google 雲端硬碟。
- ⬇️ **下載 zip**：直接下載到電腦。
- 📸 **最後一格**：印出這本 notebook 跑過的所有 State，截圖就能交作業。

Colab 斷線後 `/content/` 裡的檔案會消失，想留下來就先存。notebook 本身請用上方選單「檔案 → 在雲端硬碟中儲存副本」。

In [ ]:
#@title 💾 存到 Google 雲端硬碟（會跳出授權視窗）
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
存到雲端硬碟()

In [ ]:
#@title ⬇️ 下載 zip
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
下載壓縮檔()

In [ ]:
#@title 📸 印出整份 State 歷程（截圖交作業）
import sys; sys.path.insert(0, "/content/review-bot")
try:
    from workshop import *   # 準備、記下…（「0. 環境設定」寫好的小工具）
except ImportError:
    raise SystemExit("⚠️ 找不到環境：請先執行最上面的「0. 環境設定」那一格")
model = 準備()
印出歷程()